# 02 — Continual AHR competitive training

This notebook keeps one AHR-only training lineage. It mixes current self-play, historical AHR opponents, and GNU Go level 1 as an opponent. GNU moves are never policy labels. There is no JEPA graph.

**Inputs:** notebook 00 definitions, notebook 01 goal-conditioned AHR, frozen prior B training trajectories, and the pinned local GNU binary. **Outputs:** immutable training trajectories, full resumable checkpoints in `checkpoints/<run_id>/`, and a compact training receipt in `reports/ahr_native_v4/`.

**Default Run All:** imports and short structural checks; normally under one minute. **Recovery validation:** explicit `AHRV4_TRAIN_ACTION=validate`, several minutes. **Full training:** invoked by the detached controller; scheduled capacity 2,048 games with cumulative development stops at 128, 256, 512, 1,024, and 2,048. The first 512 games can take many hours. Only CUDA device 1 is used.

**Exact resume:** run the same `v4_run_training(PROJECT, run_id, V4TrainConfig(**saved_config), run_until_games=target)` from the controller. A later cumulative target does not change the frozen configuration. The manifest identifies the latest valid state. Invalid source/data/configuration or corrupt retained checkpoints cause an error. Checkpoints preserve optimizer, scheduler, scaler, all RNG streams used by this process, sampler/replay position, historical opponents, and the current game including a pending GNU request or verified move. Committed state resumes exactly; subsequent wall-clock search can still depend on scheduling.

The manifest keeps the latest two states and the checkpoint with lowest observed training loss. That loss checkpoint is **not** a strength promotion. Development games and final lockbox games are never imported as training. UNKNOWN cutoffs have no outcome, score, or ownership label.

In [ ]:
from pathlib import Path
import os, nbformat
PROJECT=Path(os.environ.get('AHRV4_WORKSPACE',Path.cwd())).resolve()
while not (PROJECT/'notebooks/00_instance_setup_and_recovery.ipynb').exists():
    if PROJECT.parent==PROJECT:raise RuntimeError('Run within the AHR Go workspace')
    PROJECT=PROJECT.parent
_shared=nbformat.read(PROJECT/'experiments/ahr_native_v4/00_shared.ipynb',4)
for _cell in _shared.cells:
    if _cell.cell_type=='code' and 'ahr-export' in _cell.metadata.get('tags',[]):exec(compile(_cell.source,'00_shared:ahr-export','exec'),globals())
v4_load_upstream(PROJECT,globals())
v4_import_exports(PROJECT/'experiments/ahr_native_v4/01_native_ahr_and_tactics.ipynb',globals())

## Frozen training configuration and prior training data

A four-game block contains two current-self-play games, one historical-opponent game, and one GNU game. Learner color alternates between blocks, which balances colors against GNU. Previous B trajectories supply terminal targets; their old diffuse policy targets receive zero weight until the new AHR reanalyzes a position. New GNU moves also receive zero policy weight. Imported data stay immutable and retain source hashes.

In [ ]:
from dataclasses import dataclass, asdict, field
from collections import Counter, defaultdict, OrderedDict
import copy, hashlib, json, math, os, random, subprocess, sys, time, uuid
import numpy as np
import torch

V4_TRAIN_SCHEMA='ahr_native_competitive_training_v4'
V4_TRAJECTORY_SCHEMA='ahr_native_training_trajectory_v4'
V4_TRAIN_NOTEBOOK='experiments/ahr_native_v4/02_competitive_training.ipynb'
V4_PRIOR_RUNS=('ahr-v3-B-s01','ahr-v3-B-s02','ahr-v3-B-s03')

@dataclass(frozen=True)
class V4TrainConfig:
    seed:int=2026100201
    scheduled_games:int=2048
    warmup_updates:int=256
    updates_per_game:int=16
    batch_size:int=32
    learning_rate:float=0.0002
    weight_decay:float=0.0001
    device:str='cuda:1'
    board_size:int=9
    komi:float=7.5
    max_plies:int=256
    search:dict=field(default_factory=dict)
    reanalysis_search:dict=field(default_factory=lambda:{'time_budget_s':3.0,'max_depth':5,'max_nodes':1536,'branch_width':10})
    reanalysis_per_game:int=8
    tactical_fraction:float=0.25
    reanalysis_fraction:float=0.25
    replay_capacity_games:int=4096
    opponent_snapshots:int=8
    snapshot_every_games:int=32
    exploration_start:float=0.10
    exploration_end:float=0.02
    temperature_start:float=1.0
    temperature_end:float=0.25
    temperature_opening_plies:int=20
    exploration_decay_games:int=512
    gnu_level:int=1
    gnu_budget_s:float=5.0
    gnu_command_timeout_s:float=3.0
    prior_runs:tuple=V4_PRIOR_RUNS
    checkpoint_every_events:int=32
    checkpoint_every_optimizer_steps:int=250
    checkpoint_interval_s:float=600.0
    purpose:str='go_training'
    validation_scripted:bool=False
    validation_initial_game:int=0
    def __post_init__(self):
        object.__setattr__(self,'prior_runs',tuple(self.prior_runs))
        for name in ('scheduled_games','updates_per_game','batch_size','max_plies','replay_capacity_games','opponent_snapshots','snapshot_every_games','exploration_decay_games','checkpoint_every_events','checkpoint_every_optimizer_steps'):
            if type(getattr(self,name)) is not int or getattr(self,name)<1:raise ValueError(name+' must be positive')
        if self.warmup_updates<0 or self.reanalysis_per_game<0:raise ValueError('Negative optional event count')
        if self.board_size!=9 or self.komi!=7.5:raise ValueError('This experiment fixes 9x9 and komi 7.5')
        if self.device not in ('cpu','cuda:1'):raise ValueError('Only CPU or project CUDA device 1 is authorized')
        if self.device=='cuda:1' and (not torch.cuda.is_available() or torch.cuda.device_count()<2):raise RuntimeError('CUDA device 1 unavailable')
        if not 0<=self.tactical_fraction<1 or not 0<=self.reanalysis_fraction<1 or self.tactical_fraction+self.reanalysis_fraction>=1:raise ValueError('Invalid batch mixture')
        if not 0<=self.exploration_end<=self.exploration_start<=1:raise ValueError('Invalid exploration schedule')
        if not 0<self.temperature_end<=self.temperature_start:raise ValueError('Invalid temperature schedule')
        if self.checkpoint_interval_s>600 or self.checkpoint_every_optimizer_steps>250 or self.checkpoint_every_events>32:raise ValueError('Checkpoint cadence too sparse')
        if self.gnu_level!=1 or self.gnu_command_timeout_s<=0 or self.gnu_budget_s<self.gnu_command_timeout_s:raise ValueError('Invalid GNU opponent budget')
        if self.purpose not in ('go_training','synthetic_smoke'):raise ValueError('Unknown purpose')
        if (self.validation_scripted or self.validation_initial_game) and self.purpose!='synthetic_smoke':raise ValueError('Fixtures must never masquerade as real training')
        if any(r not in V4_PRIOR_RUNS for r in self.prior_runs):raise ValueError('Only frozen prior B training runs may be imported')
        V4SearchConfig(**self.search);V4SearchConfig(**self.reanalysis_search)

def v4_training_path(project,relative):
    project=Path(project).resolve();path=(project/relative).resolve()
    if not path.is_relative_to(project):raise ValueError('Path escapes project')
    return path

def v4_training_prior(project,config):
    entries=[];manifests={}
    for run_id in config.prior_runs:
        root=Path(project)/'experiments/ahr_jepa_v3/data/trajectories'/run_id
        manifest_path=root/'manifest.json';manifest=json.loads(manifest_path.read_text())
        if manifest.get('run_id')!=run_id:raise RuntimeError('Prior archive identity changed')
        manifests[str(manifest_path.relative_to(project))]=file_sha256(manifest_path)
        training_report=json.loads((Path(project)/'reports/ahr_jepa_v3'/(run_id+'_training.json')).read_text())
        if training_report.get('status')!='COMPLETE' or training_report.get('config',{}).get('purpose')!='go_training':raise RuntimeError('Prior source is not completed real training')
        for entry in manifest['entries']:
            if entry.get('split')!='train' or entry.get('status') not in ('VERIFIED_TERMINAL','UNKNOWN'):raise RuntimeError('Nontraining or unfinished prior record')
            path=root/entry['file']
            if Path(entry['file']).name!=entry['file'] or file_sha256(path)!=entry['sha256']:raise RuntimeError('Corrupt prior trajectory')
            entries.append({**copy.deepcopy(entry),'path':str(path.relative_to(project)),'source_run':run_id,'origin':'prior_training','priority':1.0})
    return entries,{'manifests':manifests,'entries_digest':v4_digest(entries),'games':len(entries),'policy_labels':'discarded; terminal targets only until own AHR reanalysis'}

def v4_training_identity(project,config,curriculum,prior):
    return {'schema':V4_TRAIN_SCHEMA,'sources':v4_source_identity(project),'config':asdict(config),
            'curriculum':v4_digest({'train':curriculum['train'],'provenance':curriculum['provenance']}),
            'prior_training':prior,'gnu_binary':gnugo10_binary_identity(project)}

def v4_training_rng(device):
    cuda={}
    if str(device).startswith('cuda:'):cuda[str(torch.device(device).index)]=torch.cuda.get_rng_state(torch.device(device)).cpu()
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch_cpu':torch.get_rng_state(),'torch_cuda_used_devices':cuda}

def v4_restore_training_rng(rng):
    random.setstate(rng['python']);np.random.set_state(rng['numpy']);torch.set_rng_state(rng['torch_cpu'])
    for index,state in rng['torch_cuda_used_devices'].items():torch.cuda.set_rng_state(state,device=int(index))

def v4_read_training_checkpoint(run_dir,identity=None,best=False):
    run_dir=Path(run_dir);manifest_path=run_dir/'manifest.json'
    if not manifest_path.exists():
        if list(run_dir.glob('*.pt')):raise RuntimeError('Checkpoint payload exists without manifest')
        return None,None
    manifest=json.loads(manifest_path.read_text());entries=manifest.get('entries',[])
    if not entries or len({e['file'] for e in entries})!=len(entries):raise RuntimeError('Invalid checkpoint manifest')
    if manifest.get('latest')!=max(entries,key=lambda e:e['global_step'])['file']:raise RuntimeError('Latest checkpoint cursor mismatch')
    if manifest.get('best') not in {e['file'] for e in entries}:raise RuntimeError('Best checkpoint missing')
    loaded={}
    for entry in entries:
        state=_checkpoint_entry_state(run_dir,entry)
        if state is None or state.get('schema')!=V4_TRAIN_SCHEMA:raise RuntimeError('Corrupt or wrong-schema retained checkpoint')
        if identity is not None and state['identity']!=identity:raise RuntimeError('Frozen source, configuration, data, or GNU identity changed; refusing silent restart')
        if state['optimizer_step']<0 or state['global_step']<state['optimizer_step']:raise RuntimeError('Invalid checkpoint step')
        loaded[entry['file']]=(state,entry)
    return loaded[manifest['best' if best else 'latest']]


## Immutable trajectories and actual batch composition

Terminal boards are scored by two independent implementations. GNU actions have no policy target. Older B policy labels are discarded. New global policy labels come only from AHR analysis; goal-conditioned Worker labels remain separate. The sampler records categories **after** assembling the batch, including reanalysis, so replacement cannot inflate exposure counts. D4 augmentation transforms the board, history, target mask, policy and ownership together.

In [ ]:
def v4_action_transform(action,symmetry,size=9):
    if action==size*size:return action
    row,col=divmod(int(action),size)
    if symmetry>=4:col=size-1-col
    for _ in range(symmetry%4):row,col=col,size-1-row
    return row*size+col

def v4_vector_transform(values,symmetry,size=9):
    result=[0.0]*len(values)
    for action,value in enumerate(values):result[v4_action_transform(action,symmetry,size)]=value
    return result

def v4_training_augment(example,symmetry):
    row=copy.deepcopy(example);board=from_dict(row['state']);row['state']=to_dict(transform_go(board,symmetry))
    for key in ('policy_target','tactical_policy_target','ownership_target'):
        if key in row and row[key] is not None:row[key]=v4_vector_transform(row[key],symmetry,board.size)
    if 'tactical_targets' in row:row['tactical_targets']=[v4_action_transform(a,symmetry,board.size) for a in row['tactical_targets']]
    row['symmetry']=symmetry
    return row

def v4_training_archive_root(project,run_id):
    if not isinstance(run_id,str) or Path(run_id).name!=run_id or run_id in ('.','..'):raise ValueError('Simple run identifier required')
    return Path(project)/'experiments/ahr_native_v4/data/trajectories'/run_id

def v4_verify_training_game(game):
    if game.get('schema')!=V4_TRAJECTORY_SCHEMA or game.get('split')!='train':raise ValueError('Wrong training trajectory schema or split')
    board=from_dict(game['initial_state'])
    if not game['records'] and game.get('status')!='UNKNOWN':raise ValueError('Empty terminal training trajectory')
    for record in game['records']:
        if record['state']!=to_dict(board):raise RuntimeError('Training chronology mismatch')
        if record.get('policy_weight',0)>0:selfplay09_policy(board,record['policy_target'])
        if record['actor']=='gnu' and record.get('policy_weight',0)!=0:raise RuntimeError('GNU policy imitation is forbidden')
        action=record['action'];child=play(board,action)
        if not verify_go_emission(board,action,child):raise RuntimeError('Independent move verification failed')
        board=child
    if to_dict(board)!=game['state']:raise RuntimeError('Final training state mismatch')
    if game['status']=='VERIFIED_TERMINAL':
        if not board.terminal or area_score(board)!=arena_reference_score(board):raise RuntimeError('Unverified terminal target')
    elif game['status']=='UNKNOWN':
        if board.terminal:raise RuntimeError('Terminal board mislabeled UNKNOWN')
    else:raise ValueError('Cannot archive an active game')
    return board

def v4_archive_training_game(project,run_id,game):
    board=v4_verify_training_game(game);root=v4_training_archive_root(project,run_id);root.mkdir(parents=True,exist_ok=True)
    payload=(json.dumps(game,sort_keys=True,separators=(',',':'),allow_nan=False)+'\n').encode();digest=hashlib.sha256(payload).hexdigest();path=root/(digest+'.json')
    manifest_path=root/'manifest.json';manifest=json.loads(manifest_path.read_text()) if manifest_path.exists() else {'schema':V4_TRAJECTORY_SCHEMA,'run_id':run_id,'entries':[]}
    matches=[e for e in manifest['entries'] if e['game_id']==game['game_id']]
    if matches:
        if len(matches)!=1 or matches[0]['sha256']!=digest or file_sha256(path)!=digest:raise RuntimeError('Conflicting committed trajectory')
        return matches[0]
    if path.exists() and file_sha256(path)!=digest:raise RuntimeError('Trajectory corruption')
    if not path.exists():atomic_write_bytes(path,payload)
    entry={'game_id':game['game_id'],'path':str(path.relative_to(project)),'sha256':digest,'file':path.name,
           'source_run':run_id,'origin':'current_training','split':'train','status':game['status'],'plies':len(game['records']),
           'outcome':('black' if area_score(board)>0 else 'white' if area_score(board)<0 else 'draw') if board.terminal else 'UNKNOWN',
           'opponent':game['opponent_kind'],'priority':float(1+game.get('learner_stones_lost',0)/max(1,len(game['records'])))}
    manifest['entries'].append(entry);atomic_json(manifest_path,manifest)
    return entry

def v4_read_training_game(project,entry,runtime):
    cache=runtime.setdefault('game_cache',OrderedDict());key=entry['sha256']
    if key in cache:cache.move_to_end(key);return cache[key]
    path=v4_training_path(project,entry['path'])
    if file_sha256(path)!=entry['sha256']:raise RuntimeError('Immutable replay trajectory changed')
    game=json.loads(path.read_text())
    if game.get('split')!='train' or game.get('purpose')!='go_training' and game.get('purpose')!='synthetic_smoke':raise RuntimeError('Nontraining archive data')
    if game['status']!=entry['status'] or len(game['records'])!=entry['plies']:raise RuntimeError('Replay descriptor mismatch')
    final=from_dict(game['state'])
    if game['status']=='VERIFIED_TERMINAL' and (not final.terminal or area_score(final)!=arena_reference_score(final)):raise RuntimeError('Terminal score integrity failed')
    cache[key]=game
    while len(cache)>16:cache.popitem(last=False)
    return game

def v4_training_example(game,index,entry):
    if game['status']!='VERIFIED_TERMINAL':raise ValueError('UNKNOWN has no terminal labels')
    record=game['records'][index];board=from_dict(record['state']);final=from_dict(game['state'])
    weight=0.0 if entry['origin']=='prior_training' or record.get('actor')=='gnu' else float(record.get('policy_weight',0.0))
    return {'state':copy.deepcopy(record['state']),'policy_target':list(record.get('policy_target',[0.0]*82)),
            'policy_weight':weight,**terminal_go_targets(final,board.to_play),'source':entry['origin'],
            'source_game_sha256':entry['sha256'],'source_position':index,'split':'train'}

def v4_record_priority(game,index):
    record=game['records'][index];board=from_dict(record['state']);priority=1.0
    for stones,liberties in groups_and_liberties(board.board,board.size):
        if len(liberties)<=2:priority+=min(len(stones),10)*(3-len(liberties))
    # A stone cannot be suicidally removed by its own legal move. Inspect the opponent reply.
    after=game['records'][index+2]['state']['board'] if index+2<len(game['records']) else game['state']['board']
    removed=sum(1 for before,later in zip(board.board,after) if before==board.to_play and later!=before)
    priority+=4*removed
    if record['action']==81:priority+=3
    if game['status']=='VERIFIED_TERMINAL' and record.get('root_value') is not None:
        target=terminal_go_targets(from_dict(game['state']),board.to_play)['outcome_target']
        priority+=3*abs(float(record['root_value'])-target)
    return float(priority)

def v4_training_batch(project,state,config,curriculum,runtime):
    examples=[];consumed=Counter();sampler=state['sampler']
    tactical=curriculum['train'];categories=sorted({e['tactical_goal'] for e in tactical})
    bycategory={c:[e for e in tactical if e['tactical_goal']==c] for c in categories}
    entries=[e for e in state['replay_entries'][-config.replay_capacity_games:] if e['status']=='VERIFIED_TERMINAL']
    groups=defaultdict(list)
    for i,entry in enumerate(entries):
        age='recent' if i>=max(0,len(entries)*3//4) else 'older';groups[(entry['origin'],age,entry['outcome'])].append(entry)
    strata=sorted(groups);ntactical=max(1,round(config.batch_size*config.tactical_fraction))
    nrean=min(round(config.batch_size*config.reanalysis_fraction),config.batch_size-ntactical-1) if state['reanalysis'] else 0
    for index in range(config.batch_size):
        cursor=sampler['position'];sampler['position']+=1
        if index<ntactical or not entries and index>=ntactical+nrean:
            category=categories[cursor%len(categories)];row=copy.deepcopy(random.choice(bycategory[category]));label='tactical/'+category
            if row.get('policy_weight',0)!=0:raise RuntimeError('Tactical objective leaked into global policy')
        elif index<ntactical+nrean:
            row=copy.deepcopy(random.choice(state['reanalysis']));label='own_ahr_reanalysis'
        else:
            stratum=strata[cursor%len(strata)];entry=random.choice(groups[stratum]);game=v4_read_training_game(project,entry,runtime)
            position=random.randrange(len(game['records']));row=v4_training_example(game,position,entry);label='replay/'+'/'.join(stratum)
        examples.append(v4_training_augment(row,random.randrange(8)));consumed[label]+=1
    for label,count in consumed.items():sampler['consumed'][label]=sampler['consumed'].get(label,0)+count
    sampler['batches']+=1;sampler['epoch']=sampler['position']//max(1,len(tactical)+sum(e['plies'] for e in entries))
    sampler['last_batch']=dict(consumed)
    if sum(consumed.values())!=config.batch_size:raise AssertionError('Consumed sampler accounting mismatch')
    return examples


## Game, opponent and optimizer state machine

A GNU request is saved before starting the external process. Its complete chronological board, seed and actor identity are saved; the process is rebuilt from that state. The returned move is saved before application, preventing a committed move from being applied twice. GNU timeout, invalid move or max-ply cutoff ends the training attempt as UNKNOWN. AHR decisions follow the same prepare/commit structure.

Exploration decays with training progress and later game moves. Uniform exploration excludes arbitrary early passes and immediate one-liberty placements when other placements exist; the AHR search distribution is still available for strategic sacrifice or a justified pass. Reanalysis samples urgent low-liberty/capture positions, prior value mistakes and pass positions across current and older training games.

In [ ]:
def v4_training_components(project,config,model_config=None):
    if model_config is None:model,provenance=v4_model_initialization(project,device=config.device)
    else:model=V4AHR(model_config).to(config.device);provenance={}
    optimizer=torch.optim.AdamW(model.parameters(),lr=config.learning_rate,weight_decay=config.weight_decay)
    total=config.warmup_updates+config.scheduled_games*config.updates_per_game
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max(1,total),eta_min=config.learning_rate*.1)
    scaler=torch.amp.GradScaler('cuda',enabled=False)
    return (model,optimizer,scheduler,scaler),provenance

def v4_pack_training(state,components,device):
    model,optimizer,scheduler,scaler=components
    state.update(model=selfplay09_cpu_tree(model.state_dict()),optimizer=selfplay09_cpu_tree(optimizer.state_dict()),
                 scheduler=copy.deepcopy(scheduler.state_dict()),scaler=copy.deepcopy(scaler.state_dict()),rng=v4_training_rng(device))
    return state

def v4_restore_training(state,components):
    model,optimizer,scheduler,scaler=components
    model.load_state_dict(state['model'],strict=True);optimizer.load_state_dict(state['optimizer']);scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
    v4_restore_training_rng(state['rng'])

def v4_initial_training_state(project,run_id,config,identity,entries,components,provenance):
    model=components[0];weights=selfplay09_cpu_tree(model.state_dict())
    return {'schema':V4_TRAIN_SCHEMA,'run_id':run_id,'purpose':config.purpose,'identity':identity,'config':asdict(config),
            'model_config':asdict(model.config),'initialization':provenance,'global_step':0,'optimizer_step':0,'games_finished':0,
            'phase':'warmup' if config.warmup_updates else 'new_game','warmup_remaining':config.warmup_updates,'updates_remaining':0,
            'model':weights,'optimizer':None,'scheduler':None,'scaler':None,'rng':None,'current_game':None,'pending_move':None,
            'replay_entries':entries,'own_archive_entries':[],'reanalysis':[],'reanalysis_remaining':0,
            'opponents':[{'model':weights,'optimizer_step':0,'kind':'initial_ahr'}],
            'sampler':{'position':0,'batches':0,'epoch':0,'consumed':{},'last_batch':{},'reanalysis_sources':{}},
            'metrics':{'terminal_games':0,'unknown_games':0,'opponent_games':{},'verified_plies':0,'gnu_moves':0,'ahr_moves':0,
                       'gnu_failures':0,'ahr_deadline_misses':0,'reanalysis_positions':0,'captured_stones':0,'learner_stones_lost':0,'premature_terminal_games':0,
                       'best_training_loss':None,'last_update':None,'loss_history':[]},'stages_reached':[]}

def v4_training_actor(state,components,config,runtime,kind):
    if kind=='current':return components[0]
    if kind!='historical':raise ValueError('Unknown AHR actor')
    index=state['current_game']['opponent_index'];key=('historical',index,state['opponents'][index]['optimizer_step'])
    actors=runtime.setdefault('actors',{})
    if key not in actors:
        rng=v4_training_rng(config.device)
        actor=V4AHR(state['model_config']).to(config.device);actor.load_state_dict(state['opponents'][index]['model'],strict=True);actor.eval()
        v4_restore_training_rng(rng);actors[key]=actor
    return actors[key]

def v4_behavior_policy(board,policy,config,game_index,ply):
    base=selfplay09_policy(board,policy)
    progress=min(1.0,game_index/max(1,config.exploration_decay_games))
    temperature=config.temperature_start+(config.temperature_end-config.temperature_start)*progress
    if ply>=config.temperature_opening_plies:temperature=max(config.temperature_end,temperature*.5)
    positive=base>0;behavior=np.zeros_like(base);logweights=np.log(base[positive])/temperature
    behavior[positive]=np.exp(logweights-logweights.max());behavior/=behavior.sum()
    exploration=config.exploration_start+(config.exploration_end-config.exploration_start)*progress
    if ply>=80:exploration*=.25
    legal=list(legal_actions(board));placements=[]
    for action in legal:
        if action==81:continue
        child=play(board,action);stones,liberties=group_and_liberties(child.board,child.size,action)
        if len(liberties)>=2:placements.append(action)
    eligible=placements or [a for a in legal if a!=81] or legal
    if ply>=80 and 81 in legal and base[81]>.25:eligible=list(eligible)+[81]
    uniform=np.zeros_like(base);uniform[eligible]=1/len(eligible)
    behavior=(1-exploration)*behavior+exploration*uniform;behavior/=behavior.sum()
    return behavior,{'temperature':temperature,'exploration':exploration}

def v4_training_choose(project,state,components,config,runtime):
    game=state['current_game'];board=from_dict(game['state']);pending=state['pending_move']
    if pending['state_sha256']!=v4_digest(game['state']):raise RuntimeError('Pending move belongs to a different board')
    actor_kind=pending['actor']
    if actor_kind=='gnu':
        opponent=GNU10ReplayPlayer(project,level=config.gnu_level,seed=game['opponent_seed'],command_timeout_s=config.gnu_command_timeout_s)
        opponent.prepare_position(board,[],[r['action'] for r in game['records']])
        try:
            result=opponent(board,config.gnu_budget_s)
            if result.get('deadline_missed'):raise TimeoutError('GNU whole-call budget exceeded')
        except (TimeoutError,ValueError,RuntimeError) as error:
            state['metrics']['gnu_failures']+=1;game.update(status='UNKNOWN',reason='gnu_'+type(error).__name__)
            state['pending_move']=None;state['phase']='ingest';return
        policy=[0.0]*82;behavior=[0.0]*82;behavior[result['action']]=1.0
        prepared={'action':int(result['action']),'actor':'gnu','policy_target':policy,'policy_weight':0.0,'behavior_policy':behavior,
                  'root_value':None,'reference_seed':int(result['reference_seed']),'analysis':{'status':'GNU_OPPONENT_ONLY','strategic_status':'UNKNOWN'}}
    else:
        actor=v4_training_actor(state,components,config,runtime,actor_kind);actor.eval()
        if config.validation_scripted:
            policy=[0.0]*82;policy[81]=1.0;result={'action':81,'policy_target':policy,'root_value':0.0,'status':'SYNTHETIC_PASS_FIXTURE','nodes':0,'completed_depth':0,'deadline_missed':False}
            behavior=np.array(policy);schedule={'temperature':1.0,'exploration':0.0}
        else:
            with torch.inference_mode():result=v4_choose_move(actor,board,V4SearchConfig(**config.search))
            policy=selfplay09_policy(board,result['policy_target']).tolist()
            behavior,schedule=v4_behavior_policy(board,policy,config,state['games_finished'],len(game['records']))
        action=int(np.random.choice(82,p=behavior))
        state['metrics']['ahr_deadline_misses']+=int(result.get('deadline_missed',False))
        prepared={'action':action,'actor':actor_kind,'policy_target':policy,'policy_weight':0.0 if result.get('fallback_pass',False) else 1.0,
                  'behavior_policy':behavior.tolist(),'root_value':float(result['root_neural_value']) if result.get('root_neural_value') is not None else None,
                  'root_utility':float(result['root_value']) if result.get('root_value') is not None else None,
                  'analysis':{'status':result.get('status','UNKNOWN'),'strategic_status':'UNKNOWN','nodes':int(result.get('nodes',0)),
                              'completed_depth':int(result.get('completed_depth',0)),'schedule':schedule}}
    child=play(board,prepared['action'])
    if not verify_go_emission(board,prepared['action'],child):raise RuntimeError('Independent pending move verification failed')
    state['pending_move']={**pending,'result':prepared,'child':to_dict(child)};state['phase']='commit_move'

def v4_optimizer_event(project,state,components,config,curriculum,runtime):
    model,optimizer,scheduler,scaler=components;examples=v4_training_batch(project,state,config,curriculum,runtime)
    model.train();optimizer.zero_grad(set_to_none=True)
    loss,metrics=v4_learning_loss(model,examples)
    if not bool(torch.isfinite(loss)):raise RuntimeError('Nonfinite AHR learning objective')
    scaler.scale(loss).backward();scaler.unscale_(optimizer);norm=torch.nn.utils.clip_grad_norm_(model.parameters(),5.0,error_if_nonfinite=True)
    scaler.step(optimizer);scaler.update();scheduler.step();state['optimizer_step']+=1
    report={k:float(v.detach().cpu()) if torch.is_tensor(v) else copy.deepcopy(v) for k,v in metrics.items()}
    report.update(loss=float(loss.detach().cpu()),gradient_norm=float(norm.detach().cpu()),optimizer_step=state['optimizer_step'],
                  games_finished=state['games_finished'],consumed=copy.deepcopy(state['sampler']['last_batch']))
    state['metrics']['last_update']=report;state['metrics']['loss_history']=(state['metrics']['loss_history']+[report])[-2000:]
    best=state['metrics']['best_training_loss'];improved=best is None or report['loss']<best
    if improved:state['metrics']['best_training_loss']=report['loss']
    return improved

def v4_training_reanalysis(project,state,components,config,runtime):
    entries=[e for e in state['replay_entries'][-config.replay_capacity_games:] if e['status']=='VERIFIED_TERMINAL' and e['plies']]
    if not entries:state['reanalysis_remaining']=0;return
    current=[e for e in entries if e['origin']=='current_training'];older=[e for e in entries if e['origin']=='prior_training']
    pool=(current if random.random()<.5 else older) if current and older else entries
    entry=random.choices(pool,weights=[e.get('priority',1) for e in pool],k=1)[0];game=v4_read_training_game(project,entry,runtime)
    priority_key=('priority',entry['sha256']);priorities=runtime.setdefault('position_priority',{})
    if priority_key not in priorities:
        if len(priorities)>32:priorities.pop(next(iter(priorities)))
        priorities[priority_key]=[v4_record_priority(game,i) for i in range(len(game['records']))]
    index=random.choices(range(len(game['records'])),weights=priorities[priority_key],k=1)[0]
    example=v4_training_example(game,index,entry);board=from_dict(example['state']);model=components[0];model.eval()
    with torch.inference_mode():result=v4_choose_move(model,board,V4SearchConfig(**config.reanalysis_search))
    if not result.get('fallback_pass',False):
        example.update(policy_target=selfplay09_policy(board,result['policy_target']).tolist(),policy_weight=1.0,source='own_ahr_reanalysis',
                       reanalysis={'source_position':index,'source_game_sha256':entry['sha256'],'nodes':int(result.get('nodes',0)),
                                   'completed_depth':int(result.get('completed_depth',0)),'strategic_status':'UNKNOWN'})
        state['reanalysis'].append(example);state['metrics']['reanalysis_positions']+=1
        key=entry['origin'];state['sampler']['reanalysis_sources'][key]=state['sampler']['reanalysis_sources'].get(key,0)+1
    state['reanalysis_remaining']-=1

def v4_training_event(project,run_id,state,components,config,curriculum,runtime):
    improved=False
    if state['phase']=='warmup':
        improved=v4_optimizer_event(project,state,components,config,curriculum,runtime);state['warmup_remaining']-=1
        if not state['warmup_remaining']:state['phase']='new_game'
    elif state['phase']=='new_game':
        runtime.pop('actors',None)
        if state['games_finished']>=config.scheduled_games:state['phase']='complete'
        else:
            index=state['games_finished'];schedule_index=index+config.validation_initial_game
            kind=('current','current','historical','gnu')[schedule_index%4];learner_color=1 if (schedule_index//4)%2==0 else -1
            opponent_index=random.randrange(len(state['opponents'])) if kind=='historical' else None
            board=initial_state(config.board_size,config.komi)
            state['current_game']={'schema':V4_TRAJECTORY_SCHEMA,'purpose':config.purpose,'split':'train','run_id':run_id,'game_id':index,
                'initial_state':to_dict(board),'state':to_dict(board),'records':[],'status':'IN_PROGRESS','reason':None,
                'opponent_kind':kind,'opponent_index':opponent_index,'learner_color':learner_color,
                'opponent_seed':int((config.seed+1000003*index)%(2**31)),'learner_stones_lost':0}
            state['phase']='play'
    elif state['phase']=='play':
        game=state['current_game'];board=from_dict(game['state'])
        if len(game['records'])>=config.max_plies:game.update(status='UNKNOWN',reason='max_plies_cutoff');state['phase']='ingest'
        else:
            actor=game['opponent_kind'] if board.to_play!=game['learner_color'] else 'current'
            state['pending_move']={'actor':actor,'state_sha256':v4_digest(game['state']),'ply':len(game['records']),
                                   'opponent_seed':game['opponent_seed'],'result':None};state['phase']='choose_move'
    elif state['phase']=='choose_move':v4_training_choose(project,state,components,config,runtime)
    elif state['phase']=='commit_move':
        game=state['current_game'];pending=state['pending_move'];board=from_dict(game['state']);result=pending['result']
        if pending['state_sha256']!=v4_digest(game['state']) or pending['ply']!=len(game['records']):raise RuntimeError('Duplicate or stale pending move')
        child=play(board,result['action'])
        if to_dict(child)!=pending['child'] or not verify_go_emission(board,result['action'],child):raise RuntimeError('Committed child verification failed')
        removed=sum(1 for old,new in zip(board.board,child.board) if old and old!=new)
        lost=sum(1 for old,new in zip(board.board,child.board) if old==game['learner_color'] and old!=new)
        state['metrics']['captured_stones']+=removed
        game['learner_stones_lost']+=lost;state['metrics']['learner_stones_lost']+=lost
        game['records'].append({'state':to_dict(board),**result});game['state']=to_dict(child);state['pending_move']=None
        state['metrics']['verified_plies']+=1;state['metrics']['gnu_moves' if result['actor']=='gnu' else 'ahr_moves']+=1
        if child.terminal:game.update(status='VERIFIED_TERMINAL',reason='two_legal_passes');state['phase']='ingest'
        else:state['phase']='play'
    elif state['phase']=='ingest':
        game=state['current_game'];entry=v4_archive_training_game(project,run_id,game)
        if not any(e['sha256']==entry['sha256'] for e in state['own_archive_entries']):state['own_archive_entries'].append(entry);state['replay_entries'].append(entry)
        state['metrics']['terminal_games' if game['status']=='VERIFIED_TERMINAL' else 'unknown_games']+=1
        kind=game['opponent_kind'];state['metrics']['opponent_games'][kind]=state['metrics']['opponent_games'].get(kind,0)+1
        if game['status']=='VERIFIED_TERMINAL' and len(game['records'])<30:state['metrics']['premature_terminal_games']+=1
        state['games_finished']+=1;state['updates_remaining']=config.updates_per_game;state['reanalysis']=[]
        state['reanalysis_remaining']=0 if config.validation_scripted else config.reanalysis_per_game
        state['current_game']=None;state['phase']='reanalysis' if state['reanalysis_remaining'] else 'learn'
    elif state['phase']=='reanalysis':
        v4_training_reanalysis(project,state,components,config,runtime)
        if not state['reanalysis_remaining']:state['phase']='learn'
    elif state['phase']=='learn':
        improved=v4_optimizer_event(project,state,components,config,curriculum,runtime);state['updates_remaining']-=1
        if not state['updates_remaining']:
            if state['games_finished']%config.snapshot_every_games==0:
                initial=[o for o in state['opponents'] if o['kind']=='initial_ahr'];rolling=[o for o in state['opponents'] if o['kind']!='initial_ahr']
                snapshot={'model':selfplay09_cpu_tree(components[0].state_dict()),'optimizer_step':state['optimizer_step'],'kind':'historical_ahr'}
                state['opponents']=initial+(rolling+[snapshot])[-config.opponent_snapshots:]
            state['phase']='new_game'
    elif state['phase']!='complete':raise RuntimeError('Unknown training phase')
    state['global_step']+=1
    return improved


## Resume, cumulative stage boundaries, and frozen model loading

The experiment configuration fixes its full schedule once. `run_until_games` is a cumulative stopping boundary, not a new configuration. A stage pauses only after its final game, reanalysis and optimizer updates are committed. The controller can then evaluate a pinned copy and continue the same lineage. The loader exposes the exact checkpoint hash; an arena must pin that file before later training prunes old checkpoints.

In [ ]:
def v4_verify_replay_references(project,entries):
    seen=set()
    for entry in entries:
        if entry.get('split')!='train' or entry['origin'] not in ('prior_training','current_training'):raise RuntimeError('Nontraining replay reference')
        if entry['sha256'] in seen:raise RuntimeError('Duplicate replay reference')
        seen.add(entry['sha256'])
        if file_sha256(v4_training_path(project,entry['path']))!=entry['sha256']:raise RuntimeError('Corrupt replay dependency')
    return len(seen)

def v4_training_receipt(project,run_id,state,entry,target):
    complete=state['phase']=='complete';stage=state['phase']=='new_game' and state['games_finished']>=target
    metrics={k:copy.deepcopy(v) for k,v in state['metrics'].items() if k!='loss_history'}
    report={'schema':V4_TRAIN_SCHEMA,'status':'COMPLETE' if complete else 'STAGE_COMPLETE' if stage else 'RUNNING',
            'run_id':run_id,'purpose':state['purpose'],'phase':state['phase'],'games_finished':state['games_finished'],
            'completed_games':state['metrics']['terminal_games'],'unknown_games':state['metrics']['unknown_games'],
            'global_step':state['global_step'],'optimizer_steps':state['optimizer_step'],'run_until_games':target,
            'scheduled_games':state['config']['scheduled_games'],'config':state['config'],'model_config':state['model_config'],
            'checkpoint_path':str((Path('checkpoints')/run_id/entry['file'])),'checkpoint_sha256':entry['sha256'],
            'model_checkpoint':entry,'source_identity':state['identity']['sources'],'initialization':state['initialization'],
            'archive_games':len(state['own_archive_entries']),'archive_plies':sum(e['plies'] for e in state['own_archive_entries']),
            'imported_training_games':sum(e['origin']=='prior_training' for e in state['replay_entries']),
            'metrics':metrics,'loss_history_tail':state['metrics']['loss_history'][-100:],'sampler':state['sampler'],
            'stages_reached':state['stages_reached'],'current_game_plies':len(state['current_game']['records']) if state['current_game'] else None,
            'pending_actor':state['pending_move']['actor'] if state['pending_move'] else None,
            'updated_unix':time.time(),'resume_notebook':V4_TRAIN_NOTEBOOK,
            'strength_claim':'NOT_ESTABLISHED: use independent development and final arenas; training-loss best is not promotion'}
    atomic_json(v4_report_path(project,run_id+'_training.json'),report)
    return report

def v4_run_training(project,run_id,config,*,stop_after_events=None,run_until_games=None):
    project=Path(project).resolve();config=config if isinstance(config,V4TrainConfig) else V4TrainConfig(**config)
    target=config.scheduled_games if run_until_games is None else run_until_games
    if type(target) is not int or not 0<target<=config.scheduled_games:raise ValueError('Cumulative target outside frozen schedule')
    if stop_after_events is not None and (type(stop_after_events) is not int or stop_after_events<0):raise ValueError('Invalid event limit')
    v4_training_archive_root(project,run_id)
    curriculum=v4_tactical_curriculum(project)
    if not curriculum['train'] or any(e.get('split','train')!='train' for e in curriculum['train']):raise RuntimeError('Missing or contaminated tactical training curriculum')
    prior_entries,prior=v4_training_prior(project,config);identity=v4_training_identity(project,config,curriculum,prior)
    run_dir=project/'checkpoints'/run_id
    with checkpoint_lock(run_dir):
        state,entry=v4_read_training_checkpoint(run_dir,identity)
        # Configure deterministic math in EVERY fresh process, then restore checkpoint RNG below.
        v4_seed(config.seed)
        if state is None:
            components,provenance=v4_training_components(project,config)
            state=v4_initial_training_state(project,run_id,config,identity,prior_entries,components,provenance)
            entry=_save_checkpoint_unlocked(run_dir,v4_pack_training(state,components,config.device),best=True,
                resume_cell=V4_TRAIN_NOTEBOOK+': v4_run_training with saved config and cumulative run_until_games')
        else:
            v4_verify_replay_references(project,state['replay_entries']);components,_=v4_training_components(project,config,state['model_config']);v4_restore_training(state,components)
        if target<state['games_finished']:raise RuntimeError('Cannot rewind a continual lineage; load its pinned stage checkpoint for evaluation')
        first=last_event=state['global_step'];last_update=state['optimizer_step'];last_time=time.monotonic();runtime={}
        while state['phase']!='complete':
            if state['phase']=='new_game' and state['games_finished']>=target:
                if target not in state['stages_reached']:
                    state['stages_reached'].append(target);state['global_step']+=1
                    if target==config.scheduled_games:state['phase']='complete'
                break
            if stop_after_events is not None and state['global_step']-first>=stop_after_events:break
            phase=state['phase'];epoch=state['sampler']['epoch'];improved=v4_training_event(project,run_id,state,components,config,curriculum,runtime)
            due=(phase!=state['phase'] or epoch!=state['sampler']['epoch'] or improved or
                 state['global_step']-last_event>=config.checkpoint_every_events or state['optimizer_step']-last_update>=config.checkpoint_every_optimizer_steps or
                 time.monotonic()-last_time>=config.checkpoint_interval_s)
            if due:
                entry=_save_checkpoint_unlocked(run_dir,v4_pack_training(state,components,config.device),best=improved,
                    resume_cell=V4_TRAIN_NOTEBOOK+': v4_run_training with saved config and cumulative run_until_games')
                last_event=state['global_step'];last_update=state['optimizer_step'];last_time=time.monotonic();v4_training_receipt(project,run_id,state,entry,target)
        if state['global_step']!=last_event:
            entry=_save_checkpoint_unlocked(run_dir,v4_pack_training(state,components,config.device),
                resume_cell=V4_TRAIN_NOTEBOOK+': v4_run_training with saved config and cumulative run_until_games')
        return v4_training_receipt(project,run_id,state,entry,target)

def v4_load_model(project,run_id,device='cuda:1',best=False,checkpoint_path=None):
    project=Path(project).resolve();v4_training_archive_root(project,run_id)
    if device not in ('cpu','cuda:1'):raise ValueError('Use CPU or authorized CUDA device 1')
    if checkpoint_path is None:
        state,entry=v4_read_training_checkpoint(project/'checkpoints'/run_id,best=best)
        if state is None:raise FileNotFoundError('No AHR v4 training checkpoint')
        path=project/'checkpoints'/run_id/entry['file']
    else:
        path=v4_training_path(project,checkpoint_path);digest=file_sha256(path)
        state=torch.load(path,map_location='cpu',weights_only=False)
        if not isinstance(state,dict) or state.get('schema')!=V4_TRAIN_SCHEMA or state.get('run_id')!=run_id:raise RuntimeError('Wrong pinned checkpoint identity')
        entry={'file':path.name,'sha256':digest,'global_step':state['global_step']}
    if state['identity']['sources']!=v4_source_identity(project):raise RuntimeError('Checkpoint source identity differs from current frozen definitions')
    if state['purpose']!='go_training' or state['config'].get('validation_scripted'):raise RuntimeError('Synthetic fixtures cannot be loaded as trained candidates')
    model=V4AHR(state['model_config']).to(device);model.load_state_dict(state['model'],strict=True);model.eval()
    receipt={'schema':V4_TRAIN_SCHEMA,'run_id':run_id,'purpose':state['purpose'],'checkpoint_path':str(path.relative_to(project)),
             'checkpoint_sha256':entry['sha256'],'global_step':state['global_step'],'optimizer_steps':state['optimizer_step'],
             'games_finished':state['games_finished'],'config':state['config'],'model_config':state['model_config'],
             'source_identity':state['identity']['sources'],'phase':state['phase'],'initialization':state['initialization']}
    return model,receipt


## Disposable recovery and invariant checks

Validation compares uninterrupted and fresh-process interrupted runs, including interruption before a GNU request and after its verified result is saved. It exercises real GNU calls, optimizer updates and historical actors, compares model/optimizer/scheduler/scaler/RNG/sampler/replay semantics, and checks corruption, source mismatch and single-writer rejection. A second mode uses bounded real AHR search. Fixture checkpoints cannot be loaded as trained candidates.

In [ ]:
def v4_fresh_training_call(project,run_id,config,stop_after_events=None,run_until_games=None):
    project=Path(project).resolve();config=config if isinstance(config,V4TrainConfig) else V4TrainConfig(**config)
    suffix=uuid.uuid4().hex[:8];output=v4_report_path(project,run_id+'-'+suffix+'.ipynb');log=output.with_suffix('.log')
    env=os.environ.copy()
    for key in list(env):
        if key.startswith('AHRV4_'):env.pop(key)
    env.update(AHRV4_TRAIN_ACTION='worker',AHRV4_TRAIN_RUN=run_id,AHRV4_TRAIN_CONFIG=json.dumps(asdict(config)),AHRV4_WORKSPACE=str(project))
    if stop_after_events is not None:env['AHRV4_TRAIN_STOP']=str(stop_after_events)
    if run_until_games is not None:env['AHRV4_TRAIN_TARGET']=str(run_until_games)
    command=[str(project/'.venv/bin/python'),'-m','jupyter','nbconvert','--execute','--to','notebook',
             '--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=900',
             '--output',str(output),str(project/V4_TRAIN_NOTEBOOK)]
    with log.open('w') as stream:result=subprocess.run(command,cwd=project,env=env,stdout=stream,stderr=subprocess.STDOUT,timeout=1100)
    if result.returncode:raise RuntimeError('Fresh training worker failed; see '+str(log))
    return json.loads(v4_report_path(project,run_id+'_training.json').read_text())

def v4_semantic_training_state(project,state):
    semantic={k:copy.deepcopy(state[k]) for k in ('model','optimizer','scheduler','scaler','rng','sampler','metrics','optimizer_step','global_step',
                'games_finished','phase','warmup_remaining','updates_remaining','model_config','opponents','stages_reached')}
    trajectories=[]
    for entry in state['own_archive_entries']:
        game=json.loads(v4_training_path(project,entry['path']).read_text());game.pop('run_id',None);trajectories.append(game)
    semantic['trajectories']=trajectories
    semantic['prior_training']=[e for e in state['replay_entries'] if e['origin']=='prior_training']
    return semantic

def v4_recovery_pair(project,device='cuda:1',real_search=False):
    project=Path(project);prefix='ahr-v4-recovery-'+('real-' if real_search else 'scripted-')+uuid.uuid4().hex[:8]
    search={'time_budget_s':30.,'max_depth':1,'max_nodes':8,'branch_width':2,'tactical_width':2,'quiescence_extensions':0}
    config=V4TrainConfig(seed=2026100207,scheduled_games=2,warmup_updates=1,updates_per_game=2,batch_size=8,device=device,
        max_plies=4,search=search,reanalysis_search=search,reanalysis_per_game=0,snapshot_every_games=1,
        purpose='synthetic_smoke',validation_scripted=not real_search,validation_initial_game=3,prior_runs=('ahr-v3-B-s01',))
    full=prefix+'-full';resume=prefix+'-resume'
    v4_fresh_training_call(project,full,config)
    v4_fresh_training_call(project,resume,config,6)
    before,_=v4_read_training_checkpoint(project/'checkpoints'/resume)
    if before['phase']!='choose_move' or before['pending_move']['actor']!='gnu' or before['pending_move']['result'] is not None:
        raise AssertionError('Recovery did not stop before GNU request')
    v4_fresh_training_call(project,resume,config,1)
    after,_=v4_read_training_checkpoint(project/'checkpoints'/resume)
    if after['phase']!='commit_move' or after['pending_move']['actor']!='gnu' or after['pending_move']['result'] is None:
        raise AssertionError('Recovery did not stop after saved GNU result')
    if len(before['current_game']['records'])!=len(after['current_game']['records']):raise AssertionError('GNU pending result applied before commit')
    v4_fresh_training_call(project,resume,config)
    left,_=v4_read_training_checkpoint(project/'checkpoints'/full);right,_=v4_read_training_checkpoint(project/'checkpoints'/resume)
    if not nested_equal(v4_semantic_training_state(project,left),v4_semantic_training_state(project,right)):
        keys=[k for k in v4_semantic_training_state(project,left) if not nested_equal(v4_semantic_training_state(project,left)[k],v4_semantic_training_state(project,right)[k])]
        raise AssertionError('Recovery mismatch in '+str(keys))
    if left['optimizer_step']!=5 or left['metrics']['gnu_moves']<1:raise AssertionError('Recovery lacked real GNU and optimizer coverage')
    rejected={};run_dir=project/'checkpoints'/resume
    try:v4_read_training_checkpoint(run_dir,{'changed':'identity'})
    except RuntimeError:rejected['source_mismatch']=True
    with checkpoint_lock(run_dir):
        try:
            with checkpoint_lock(run_dir):pass
        except RuntimeError:rejected['second_writer']=True
    corrupt=project/'checkpoints'/(prefix+'-corrupt');corrupt.mkdir(parents=True,exist_ok=False)
    manifest=json.loads((run_dir/'manifest.json').read_text());atomic_json(corrupt/'manifest.json',manifest)
    try:v4_read_training_checkpoint(corrupt)
    except RuntimeError:rejected['missing_payload']=True
    try:v4_load_model(project,resume,device=device)
    except RuntimeError:rejected['synthetic_candidate']=True
    if len(rejected)!=4:raise AssertionError('Recovery rejection contract incomplete')
    manifest=json.loads((run_dir/'manifest.json').read_text())
    if len(manifest['entries'])<2 or manifest['best'] not in {e['file'] for e in manifest['entries']}:raise AssertionError('Retention contract failed')
    return {'status':'PASS','device':device,'real_ahr_search':real_search,'real_gnu_moves':left['metrics']['gnu_moves'],
            'optimizer_steps':left['optimizer_step'],'fresh_process':True,'exact_state':True,'pending_gnu_request_and_result':True,
            'runs':{'full':full,'resume':resume},'rejections':rejected,'retained':len(manifest['entries']),
            'source_identity':v4_source_identity(project),'config':asdict(config)}

def v4_validate_recovery(project,device='cuda:1'):
    started=time.time();sources=v4_source_identity(project)
    checks=[v4_recovery_pair(project,device,False),v4_recovery_pair(project,device,True)]
    if v4_source_identity(project)!=sources:raise RuntimeError('Sources changed during recovery validation')
    receipt={'status':'PASS','schema':V4_TRAIN_SCHEMA,'source_identity':sources,'checks':checks,
             'exact_resume':True,'real_gnu_recovery':True,'actual_ahr_search':True,'device':device,'elapsed_s':time.time()-started}
    atomic_json(v4_report_path(project,'training_recovery.json'),receipt)
    return receipt

def v4_training_structure_checks(project):
    curriculum=v4_tactical_curriculum(project);example=curriculum['train'][0]
    for symmetry in range(8):
        row=v4_training_augment(example,symmetry);board=from_dict(row['state'])
        if any(p>0 and a not in legal_actions(board) for a,p in enumerate(row['tactical_policy_target'])):raise AssertionError('D4 target legality failure')
        if row['policy_weight']!=0 or row['tactical_goal']!=example['tactical_goal']:raise AssertionError('Tactical/global separation failure')
    schedule=[('current','current','historical','gnu')[i%4] for i in range(512)]
    colors=[1 if i//4%2==0 else -1 for i in range(512) if schedule[i]=='gnu']
    if Counter(schedule)!={'current':256,'historical':128,'gnu':128} or Counter(colors)!={1:64,-1:64}:raise AssertionError('Opponent/color balance failure')
    report={'status':'PASS','training_steps':0,'curriculum_training_cases':len(curriculum['train']),
            'opponent_mix_first_512':dict(Counter(schedule)),'gnu_colors_first_512':dict(Counter(colors)),
            'tactical_d4_legality':True,'gnu_policy_labels':False,'jepa_graph':False,
            'long_training':'Explicit detached controller only','recovery':'Run AHRV4_TRAIN_ACTION=validate before long work'}
    atomic_json(v4_report_path(project,'training_structure.json'),report);return report


## Execute the selected bounded action

Default execution verifies structure only. The controller sets the worker configuration explicitly. The recovery action launches disposable fresh kernels and saves its source-bound receipt. No training is silently started by importing tagged definitions.

In [ ]:
ACTION=os.environ.get('AHRV4_TRAIN_ACTION','inspect')
if ACTION=='worker':
    _config=V4TrainConfig(**json.loads(os.environ['AHRV4_TRAIN_CONFIG']))
    _stop=int(os.environ['AHRV4_TRAIN_STOP']) if 'AHRV4_TRAIN_STOP' in os.environ else None
    _target=int(os.environ['AHRV4_TRAIN_TARGET']) if 'AHRV4_TRAIN_TARGET' in os.environ else None
    _receipt=v4_run_training(PROJECT,os.environ['AHRV4_TRAIN_RUN'],_config,stop_after_events=_stop,run_until_games=_target)
    print(json.dumps({k:_receipt[k] for k in ('status','run_id','games_finished','optimizer_steps','phase','checkpoint_sha256')},indent=2))
elif ACTION=='validate':
    _receipt=v4_validate_recovery(PROJECT,device=os.environ.get('AHRV4_TRAIN_DEVICE','cuda:1'))
    print(json.dumps({'status':_receipt['status'],'exact_resume':_receipt['exact_resume'],'real_gnu_recovery':_receipt['real_gnu_recovery'],'elapsed_s':_receipt['elapsed_s']},indent=2))
elif ACTION=='inspect':
    _receipt=v4_training_structure_checks(PROJECT)
    print(json.dumps(_receipt,indent=2))
else:raise ValueError('Unknown training notebook action')